<a href="https://colab.research.google.com/github/bajon1/Deep-learning-lab/blob/main/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openml optuna torch torchvision scikit-learn matplotlib numpy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openml
import optuna
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available:   {torch.cuda.is_available()}")
print(f"Device:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Lab 4: MLP Architecture Optimization

# Task 1: Regression – network depth vs. width

#### Goal: Investigate the effect of the number of hidden layers and the number of neurons per layer on regression quality for a real dataset.

1. Load a real regression dataset from OpenML (e.g. a small/medium tabular dataset with one continuous target variable).
2. Split the data into training, validation, and test sets (e.g. 60/20/20 using train_test_split).
3. A regression MLP model in PyTorch as a class inheriting from nn.Module, using nn.ModuleList to store hidden layers — architecture: input → hidden_dim → … → hidden_dim → output (1 neuron for regression), with ReLU nonlinear activation after each hidden layer.
4. Train several architecture variants under the same training setup (MSELoss, Adam optimizer, fixed learning rate, fixed number of epochs, fixed batch size): variable depth at fixed width — n_hidden ∈ {1, 2, 4} with hidden_dim = 64; variable width at fixed depth — hidden_dim ∈ {16, 64, 256} with n_hidden = 2.
5. Save the best model checkpoint and compare validation MSE across all variants.
6. Compare test MSE across all variants.
Assignment: Implement an MLP with nn.ModuleList for a regression problem on a real OpenML dataset and compare the effect of depth and width on validation error. Explain why nn.ModuleList is used instead of a plain Python list.

# Task 2: Regression – automatic depth and width optimization

####Goal: Use the Optuna library to select the optimal network depth and width.

1. For the data from Task 1, define an objective function for hyperparameter optimization using Optuna with 5-fold cross-validation.
2. Train five models with the optimal architecture (one per fold of the 5-fold cross-validation).
3. Run inference on the test set using an ensemble of the five trained models.

Assignment: Implement a function for automatic MLP architecture optimization using Optuna.

# Task 3: Multiclass classification – architecture and activation search

#### Goal: Optimize the MLP architecture for a multiclass classification task on a real dataset.

1. Select and load a multiclass classification dataset from OpenML (at least 3 classes, approximately 1,000–10,000 samples, moderate number of features).
2. Preprocessing: one-hot encode categorical features; standardize numerical features; split into train+validation and test sets (e.g. 80/20, stratified by class label).
3. An MLP in PyTorch using nn.ModuleList for hidden layers, parameterized by: number of hidden layers (n_hidden), hidden layer size (hidden_dim), and activation function (ReLU or Tanh).
4. Optimize the above hyperparameters using Optuna.
5. Extract the three best hyperparameter configurations from the optimization results.
6. For each of the three configurations: train five models using 5-fold cross-validation.
7. Evaluate all three configurations on the test set using CrossEntropyLoss and accuracy; save the results.

Assignment: Build and train the three most optimal MLP variants with nn.ModuleList for multiclass classification. Compare classification quality on the test set.

# Task 4: Multi-label classification – independent output heads in ModuleList

#### Goal: Investigate the difference between a single shared output layer and independent per-label heads in a multi-label classification task.

1. Load a multi-label classification dataset from OpenML, e.g.:

```python
dataset = openml.datasets.get_dataset(312)
X, _, _, _ = dataset.get_data(target=None)
label_cols = ["Beach", "Sunset", "FallFoliage", "Field", "Mountain", "Urban"]
attr_cols = [col for col in X.columns if 'attr' in col]
y = X[label_cols].values.astype(np.float32)
X = X[attr_cols]
```

2. Data preparation: convert labels to a 0/1 matrix of shape [n_samples, n_labels]; split into training, validation, and test sets.
3. Baseline model (shared output): MLP with nn.ModuleList for the shared network body; one linear output layer nn.Linear(hidden_dim, n_labels); sigmoid activation on output, BCEWithLogitsLoss as the loss function.
4. Multi-head model (independent branches): same shared body as above; self.heads = nn.ModuleList([nn.Linear(hidden_dim, 1) for _ in range(n_labels)]); in the forward method, pass through the body, then through each head separately, and concatenate the results along the feature dimension.
5. Train both models with identical hyperparameters (same optimizer, number of epochs, batch size) and compare multi-label metrics — Hamming loss, micro-F1, macro-F1 — on the validation and test sets.

Assignment: Implement two MLP variants with nn.ModuleList for multi-label classification and compare whether the independent-heads architecture improves prediction quality over the shared output layer.

# Task 5: Fixed parameter budget – different network shapes

#### Goal: Investigate how network shape (depth vs. width) affects model quality when the total parameter count is held constant.

1. Choose one of the previous tasks (regression or multiclass classification) and use the same dataset.
2. Set an approximate parameter budget (e.g. ~50,000 parameters).
3. Design at least three MLP architectures that fit within the budget (±10%) but differ in shape: deep-narrow (more layers, smaller hidden_dim); shallow-wide (1–2 layers, large hidden_dim); "pyramid" (increasing or decreasing layer sizes, e.g. 32 → 64 → 32).
4. Compute and print the exact parameter count for each architecture.
5. Train each architecture under an identical setup (same number of epochs, batch size, optimizer, learning rate) and compare: training and validation loss curves; final validation results (MSE or accuracy).

Assignment: Build several MLP architectures with a similar parameter count and determine which network shape — deep-narrow, shallow-wide, or pyramid — achieves the best results.